# Proyecto final · Analítica para los Negocios II

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hizocar/analitica-negocios-2/blob/main/docs/notebooks/proyecto_final_plantilla.ipynb)

| | |
|---|---|
| **Título del proyecto** | *(escribe aquí tu recomendación en una frase, no el tema)* |
| **Caso** | FinTrust (reclamos) / MercaSur (reseñas) — *borra el que no corresponda* |
| **Equipo** | Nombre 1 · Nombre 2 · Nombre 3 |
| **Fecha de entrega** | Domingo 29 de noviembre de 2026, 23:59 (buzón) |
| **Presentación** | Jueves 3 de diciembre de 2026 · 10 min + 5 min de preguntas |

> **Cómo usar esta plantilla:** haz una copia en tu Drive (*Archivo → Guardar una copia en Drive*). Cada sección trae
> instrucciones (en cursiva, bórralas al final) y celdas de código mínimas. Las celdas marcadas con `# TODO` las
> completas con el trabajo de tus sprints. Antes de entregar: *Entorno de ejecución → Reiniciar y ejecutar todo*,
> y verifica que **corre de principio a fin sin errores**. La rúbrica está en la página
> *Proyecto → Enunciado y rúbrica* del sitio del curso.

**Resumen ejecutivo (5 líneas máximo):** *Escríbelo al final. Recomendación con una cifra, los tres argumentos
(funciona · vale la pena · es seguro) y la decisión que pides.*

## 0. Configuración

*Instala solo lo que falta, define el caso y carga las claves desde los Secrets de Colab (ícono 🔑 de la barra
lateral). **Nunca** pegues una clave de API en el código.*

In [ ]:
# Descomenta la instalación cuando llegues a las secciones 5–7 (embeddings, LLM y RAG)
# !pip -q install -U sentence-transformers google-genai

CASO = "fintrust"  # o "mercasur"
BASE = "https://raw.githubusercontent.com/hizocar/analitica-negocios-2/main/docs/datasets/"

import os
import re
import time
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

CONFIG = {
    "fintrust": {"archivo": "reclamos_fintrust.csv", "etiqueta": "categoria",
                 "clases": ["Fraude", "Cobro indebido", "App y canales digitales", "Crédito y tasas",
                            "Atención al cliente", "Información y trámites"],
                 "politicas": "politicas_fintrust.csv", "preguntas": "preguntas_eval_fintrust.csv"},
    "mercasur": {"archivo": "resenas_mercasur.csv", "etiqueta": "aspecto",
                 "clases": ["Despacho", "Calidad del producto", "Precio", "Atención",
                            "Devoluciones y garantía", "Pago"],
                 "politicas": "politicas_mercasur.csv", "preguntas": "preguntas_eval_mercasur.csv"},
}[CASO]
SEMILLA = 42
print("Caso:", CASO)

## 1. Pregunta de negocio

*Escribe en 4–6 líneas:*

- *¿Qué problema de la empresa resuelves? (usa el formato SCQA: situación, complicación, pregunta)*
- *¿Quién toma la decisión y qué decisión es?*
- *¿Cómo vas a medir el éxito? (métrica técnica **y** métrica de negocio, por ejemplo «% de reclamos derivados al
  área correcta» y «horas liberadas al mes»)*

## 2. Datos y limpieza

*Describe la auditoría de calidad: duplicados, nulos, textos vacíos, categorías mal escritas, valores imposibles y
datos personales. Reporta **cuántas filas** afectó cada problema y qué decidiste hacer.*

La función `limpiar` ya funciona: quita duplicados exactos de texto **antes** de separar train y test (para evitar
fuga de información), elimina textos vacíos y normaliza las categorías.

In [ ]:
def _clave(s):
    """Minúsculas, sin tildes y sin espacios sobrantes."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip().lower()

ALIAS = {"cobros indebidos": "cobro indebido", "calidad producto": "calidad del producto", "pagos": "pago"}

def limpiar(df, etiqueta, clases):
    oficial = {_clave(c): c for c in clases}
    df = df.copy()
    df["texto"] = df["texto"].fillna("").astype(str).str.strip()
    df = df.drop_duplicates(subset="texto")
    df = df[df["texto"].str.len() > 15]
    df["etiqueta"] = df[etiqueta].map(_clave).replace(ALIAS).map(oficial)
    return df.dropna(subset=["etiqueta"]).reset_index(drop=True)

crudo = pd.read_csv(BASE + CONFIG["archivo"])
df = limpiar(crudo, CONFIG["etiqueta"], CONFIG["clases"])

auditoria = pd.Series({
    "filas crudas": len(crudo),
    "duplicados exactos (texto)": int(crudo["texto"].duplicated().sum()),
    "textos vacíos o muy cortos": int((crudo["texto"].fillna("").str.strip().str.len() <= 15).sum()),
    "etiquetas con variantes de escritura": int((~crudo[CONFIG["etiqueta"]].isin(CONFIG["clases"])).sum()),
    "filas limpias": len(df),
})
auditoria

In [ ]:
# TODO: completa la auditoría con los problemas propios de tu caso
#  - FinTrust: monto_reclamado negativo, dias_resolucion imposibles (-3, 999, 540), urgencia nula, canal con variantes
#  - MercaSur: rating nulo o imposible (0, 6, 10, -1), canal_compra con variantes
#  - Ambos: textos con datos personales (RUT, teléfono, correo, nombre)

In [ ]:
# Separación train / test (estratificada). Usa SIEMPRE este mismo test para comparar todos los enfoques.
train, test = train_test_split(df, test_size=0.2, stratify=df["etiqueta"], random_state=SEMILLA)
# Subconjunto fijo de 100 casos de test para el LLM (Sprint 3) y para comparar todos en igualdad de condiciones
test_100 = test.sample(100, random_state=SEMILLA)
print(f"train: {len(train):,} | test: {len(test):,} | test_100: {len(test_100)}")

## 3. Análisis exploratorio (EDA)

*Muestra y comenta: distribución de clases (¿hay desbalance?), largo de los textos por clase, palabras más
frecuentes por clase y 2–3 hallazgos que importen para el negocio. Cada gráfico con un título que diga qué concluir.*

In [ ]:
print(df["etiqueta"].value_counts())
df["largo"] = df["texto"].str.split().str.len()
df.groupby("etiqueta")["largo"].describe().round(1)

In [ ]:
# TODO: gráfico de barras de clases con título de acción
# TODO: palabras más frecuentes por clase (CountVectorizer o collections.Counter)

## 4. Baselines (Sprint 1)

*TF-IDF + regresión logística y red neuronal MLP (Keras). Reporta F1 macro, accuracy y matriz de confusión en el
test. Comenta qué categorías se confunden y por qué.*

La función `registrar` guarda cada resultado en una lista para armar la tabla comparativa de la sección 8.

In [ ]:
resultados = []

def registrar(enfoque, y_true, y_pred, costo_usd_1000=0.0, latencia_s=np.nan, nota=""):
    """Guarda las métricas de un enfoque para la tabla comparativa final."""
    resultados.append({"enfoque": enfoque, "n_test": len(y_true),
                       "accuracy": accuracy_score(y_true, y_pred),
                       "f1_macro": f1_score(y_true, y_pred, average="macro"),
                       "costo_usd_1000": costo_usd_1000, "latencia_s": latencia_s, "nota": nota})

# TODO: TF-IDF + logística
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# ...
# registrar("TF-IDF + logística", test["etiqueta"], pred_tfidf, latencia_s=...)

In [ ]:
# TODO: red neuronal MLP con Keras 3 (import keras)
# registrar("MLP (Keras)", test["etiqueta"], pred_mlp, latencia_s=...)

## 5. Embeddings (Sprint 2)

*Embeddings con `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, clasificador sobre embeddings vs.
baselines (incluye la curva con 50 / 200 / 1.000 ejemplos), buscador semántico con 5 consultas de negocio y mapa de
temas (clustering con un nombre para cada cluster).*

In [ ]:
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# TODO:
# from sentence_transformers import SentenceTransformer
# st = SentenceTransformer(MODELO_EMB)
# E_train = st.encode(train["texto"].tolist(), batch_size=64, show_progress_bar=True)
# ...
# registrar("Embeddings + logística", test["etiqueta"], pred_emb, latencia_s=...)

## 6. LLM (Sprint 3)

*Clasificación zero-shot y few-shot sobre `test_100`, extracción estructurada en JSON, comparación de calidad,
costo y latencia, análisis de errores y **anonimización** de datos personales antes de enviar a la API.*

**Importante:** guarda las predicciones del LLM en un CSV apenas las obtengas. Así, si se acaba la cuota o no hay
clave, el notebook carga el CSV y sigue corriendo de principio a fin.

In [ ]:
try:
    from google import genai
    from google.genai import types
except ImportError:
    genai = None
    print("Falta google-genai: descomenta el !pip install de la sección 0.")

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

MODELO = "gemini-2.5-flash"   # revisa el nombre vigente en https://ai.google.dev/gemini-api/docs/models
client = genai.Client(api_key=GEMINI_API_KEY) if (genai and GEMINI_API_KEY) else None

CACHE_LLM = f"predicciones_llm_{CASO}.csv"
if os.path.exists(CACHE_LLM):
    pred_llm = pd.read_csv(CACHE_LLM)
    print(f"Predicciones del LLM cargadas desde {CACHE_LLM} ({len(pred_llm)} filas).")
elif client is None:
    print("Sin clave de Gemini: configura GEMINI_API_KEY en los Secrets de Colab (ver la guía «API de Gemini» "
          "del sitio). Mientras tanto, esta sección se salta.")
else:
    print("Cliente listo. Completa el TODO de abajo para clasificar test_100.")

In [ ]:
def anonimizar(texto):
    """Reemplaza datos personales antes de enviar a la API."""
    patrones = {"RUT": r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b", "TELEFONO": r"\+?56\s?9\s?\d{4}\s?\d{4}",
                "CORREO": r"[\w.+-]+@[\w-]+\.[\w.]+",
                "NOMBRE": r"\bAtte\.?\s+[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+\s+[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+"}
    for etiqueta, patron in patrones.items():
        texto = re.sub(patron, f"[{etiqueta}]", texto, flags=re.IGNORECASE)
    return texto

# TODO: zero-shot y few-shot con salida estructurada (response_schema con un Enum de clases)
# for texto in test_100["texto"].map(anonimizar): ...  (usa time.sleep y reintentos por los límites de la API)
# Guarda: pd.DataFrame({"id": ..., "pred": ..., "tokens_in": ..., "tokens_out": ..., "latencia_s": ...}).to_csv(CACHE_LLM, index=False)
# registrar("LLM few-shot", test_100["etiqueta"], pred_fewshot, costo_usd_1000=..., latencia_s=...)

## 7. Asistente RAG

*Asistente que responde preguntas sobre las políticas del caso: fragmentación, embeddings, recuperación de los
k fragmentos más parecidos y generación con el LLM **citando la fuente**. Evalúalo con las preguntas de evaluación
(incluida la pregunta trampa con `doc_id = NINGUNO`, donde debe decir que no sabe). Reporta % de respuestas correctas
y % de recuperación correcta del documento.*

In [ ]:
politicas = pd.read_csv(BASE + CONFIG["politicas"])
preguntas = pd.read_csv(BASE + CONFIG["preguntas"])
print(f"Políticas: {len(politicas)} | preguntas de evaluación: {len(preguntas)} "
      f"(trampa: {int((preguntas['doc_id'] == 'NINGUNO').sum())})")
preguntas.head(3)

In [ ]:
# TODO: fragmentar políticas, indexar con embeddings, recuperar top-k y generar con el LLM
# TODO: evaluar sobre `preguntas` y guardar una tabla con pregunta, respuesta, doc recuperado y si es correcta

## 8. Comparación final

*Tabla con todos los enfoques evaluados **sobre el mismo test** (o el mismo `test_100`). Agrega intervalos bootstrap
o McNemar para las diferencias que uses para decidir (ver el notebook de la Clase 5) y una matriz de decisión con
pesos explícitos.*

In [ ]:
tabla_final = pd.DataFrame(resultados)
if tabla_final.empty:
    print("Aún no registras resultados: usa registrar(...) en las secciones 4–6.")
else:
    display(tabla_final.sort_values("f1_macro", ascending=False).round(3))

In [ ]:
# TODO: bootstrap pareado de la diferencia entre tu mejor modelo y el LLM (en test_100)
# TODO: matriz de decisión ponderada (puntajes 1–5 por criterio y pesos del negocio)

## 9. Recomendación y valor de negocio

*Tu recomendación en una frase con una cifra. Luego el ROI en tres escenarios (pesimista, base, optimista) con
supuestos visibles y justificados, payback y error máximo tolerable. Incluye el gráfico que irá en tu lámina.*

In [ ]:
# Supuestos (TODO: ajústalos a tu caso y justifica cada uno en el texto)
esc = pd.DataFrame({
    "volumen_mes":       [4000, 6000, 8000],
    "min_por_caso":      [4, 6, 8],
    "costo_hora":        [10000, 12000, 14000],
    "pct_automatizable": [0.50, 0.70, 0.80],
    "tasa_error":        [0.08, 0.04, 0.03],
    "costo_error":       [25000, 15000, 10000],
    "api_por_1000":      [3000, 1000, 500],
    "mantencion_mes":    [900000, 600000, 400000],
}, index=["pesimista", "base", "optimista"])
INVERSION = 8_000_000

e = esc.copy()
e["casos_auto"] = e.volumen_mes * e.pct_automatizable
e["ahorro"] = e.casos_auto * e.min_por_caso / 60 * e.costo_hora
e["costo_errores"] = e.casos_auto * e.tasa_error * e.costo_error
e["neto_mes"] = e.ahorro - e.costo_errores - e.volumen_mes / 1000 * e.api_por_1000 - e.mantencion_mes
e["payback_meses"] = np.where(e.neto_mes > 0, INVERSION / e.neto_mes, np.inf)
e[["ahorro", "costo_errores", "neto_mes", "payback_meses"]].round(1)

## 10. Riesgos y limitaciones

*Para cada riesgo: cómo se ve en tu caso, qué hiciste o qué propones. Considera: datos personales y anonimización
(Ley 21.719, vigente desde el 1 de diciembre de 2026), prompt injection, alucinaciones del RAG, desempeño por
segmento, humano en el ciclo, monitoreo y deriva. Incluye también las **limitaciones** de tu análisis (datos
sintéticos, tamaño del test del LLM, ruido de etiquetas…).*

In [ ]:
checklist = pd.DataFrame([
    ["Datos personales", "TODO", "❌"],
    ["Prompt injection", "TODO", "❌"],
    ["Alucinaciones (RAG)", "TODO", "❌"],
    ["Desempeño por segmento", "TODO", "❌"],
    ["Humano en el ciclo", "TODO", "❌"],
    ["Monitoreo y deriva", "TODO", "❌"],
], columns=["riesgo", "mitigación en nuestro proyecto", "estado"])
checklist

## 11. Conclusiones

*Tres a cinco viñetas:*

- *La recomendación (repetida) y la decisión concreta que pides (por ejemplo, un piloto acotado y medible).*
- *Qué aprendiste sobre cuándo convienen los modelos clásicos, los embeddings y los LLM en este caso.*
- *Próximos pasos: qué medirías o mejorarías con más tiempo o datos.*

---
**Checklist antes de subir al buzón:** ☐ corre de principio a fin · ☐ sin claves en el código · ☐ todas las secciones
con texto · ☐ tabla comparativa sobre el mismo test · ☐ RAG evaluado con la pregunta trampa · ☐ ROI con escenarios ·
☐ riesgos con mitigación · ☐ láminas en PDF adjuntas.